In [ ]:
import nasdaqdatalink
import logging
nasdaqdatalink.ApiConfig.verify_ssl = False
nasdaqdatalink.ApiConfig.api_key = "5Mz_mm1sWnrxzrVJXVvz"

mydata = nasdaqdatalink.get("CHRIS/CBOE_VX1")

The Carhart four-factor model can also be applied to the VIX index. However, since the VIX is not a stock, we cannot use the same factors as we would for individual stocks. Instead, we need to use factors that are relevant for the volatility index.

In this case, we can use the market returns, the term structure of volatility (measured as the difference between the implied volatility of near-term and long-term VIX futures), the skewness of S&P 500 returns, and the momentum of VIX futures returns.

Assuming you have a dataset with the following columns: date, VIX returns, market returns, term structure, skewness, and momentum, the following code can be used to compute the Carhart four-factor model for the VIX index:

This code downloads the daily adjusted close prices for the S&P 500 index and the VIX index using Yahoo Finance API. It also downloads the VIX term structure data and S&P 500 skewness data from Quandl. Finally, it computes the momentum of VIX futures returns using a rolling window of 10 days. The resulting dataframe includes the requested columns and is printed to the console. Note that you will need to have valid API keys for Yahoo Finance and Quandl to run this code.

In [ ]:
mydata

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import quandl

# Set start and end dates for data collection
start_date = '2020-06-01'
end_date = '2021-06-01'

spy_data=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)
vix_data = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

spy_data = spy_data[start_date:end_date]
vix_data = vix_data[start_date:end_date]

spy_data["close(t-1)"] = spy_data["close"].shift(periods=1)
spy_data["close(t-1)"].ffill(inplace=True)

spy_data["spx return"]=np.log(spy_data["close"]/spy_data["close(t-1)"])
spy_data["spx return"].ffill(inplace=True)

vix_data["close(t-1)"] = vix_data["close"].shift(periods=1)
vix_data["close(t-1)"].ffill(inplace=True)

vix_data["vix return"]=np.log(vix_data["close"]/vix_data["close(t-1)"])
vix_data["vix return"].ffill(inplace=True)

spy_data = spy_data["2020-06-01":]
# Download VIX term structure data from Quandl
quandl.ApiConfig.api_key = "YOUR_API_KEY"
term_structure_data = quandl.get("CHRIS/CBOE_VX1", start_date=start_date, end_date=end_date,authtoken='5Mz_mm1sWnrxzrVJXVvz')
term_structure = (term_structure_data['Close'] - term_structure_data['Settle']).pct_change()

# Download S&P 500 skewness data from FRED
fred_data = pd.read_csv('/Users/Farid Srouji/Documents/Research/master/Data/CSV/CBOE SKEW Historical Data.csv', compression=None) 
fred_data.index = pd.to_datetime(fred_data.Date)
fred_data = fred_data[start_date:end_date]

#quandl.get("FRED/SP500SKEW", start_date=start_date, end_date=end_date,authtoken='5Mz_mm1sWnrxzrVJXVvz')
skewness = fred_data['Price'].pct_change()

# Compute momentum of VIX futures returns
vix_futures_data = quandl.get("CHRIS/CBOE_VX1", start_date=start_date, end_date=end_date,authtoken='5Mz_mm1sWnrxzrVJXVvz')
vix_futures_returns = vix_futures_data['Close'].pct_change()
momentum = vix_futures_returns.rolling(window=10).mean().pct_change()

# Combine data into a single dataframe
df = pd.DataFrame({
    'date': spy_data.index,
    'VIX returns': vix_data["vix return"],
    'market returns': spy_data["spx return"],
    'term structure': term_structure,
    'skewness': skewness,
    'momentum': momentum
}).dropna()

# Print the dataframe
print(df.head())

In [ ]:
df["momentum"].plot(title="VIX Momentum")

In [ ]:
df.to_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/VIXFactors.pkl')

In [ ]:
tr=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/Data/PKL/TreasuryRates.pkl', compression=None)


In [ ]:
tr.sort_index()["2020-06-01":"2022-06-01"]

In [ ]:
df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

# Load data

# Create 3D plot of VIX variables
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
x = df['term structure']
y = df['skewness']
z = df['momentum']
ax.scatter(x, y, z, c='b', marker='o')
ax.set_xlabel('Term Structure')
ax.set_ylabel('Skewness')
ax.set_zlabel('VIX returns')
ax.set_title('3D Plot of VIX Variables')
plt.show()